# Analisis Performa Model Naive Bayes Berbasis Ekstraksi Fitur TF-IDF dan Skip-Gram pada Klasifikasi Teks Berita

### Import Library


In [1]:
import pandas as pd
import numpy as np
import re
import os
import pickle
import warnings

warnings.filterwarnings("ignore")

print("Library dasar berhasil di-import.")

Library dasar berhasil di-import.


### Membaca Dataset

In [2]:
file_path = "berita_detik_200_artikel.xlsx"

df = pd.read_excel(file_path)

print("Dataset berhasil dibaca.")
print("Ukuran dataset:", df.shape)
print("\nNama kolom:")
print(df.columns.tolist())

print("\n5 data pertama:")
display(df.head())

Dataset berhasil dibaca.
Ukuran dataset: (200, 3)

Nama kolom:
['id', 'isi_berita', 'label']

5 data pertama:


,id,isi_berita,label
0,1,Kejuaraan Nasional Gokart 2026 yang berlangsun...,sport
1,2,PT Bank Negara Indonesia (Persero) Tbk atau BN...,sport
2,3,"Pelatih Timnas voli putra Indonesia, Reidel To...",sport
3,4,Kontingen Indonesia bersiap menatap Asian Game...,sport
4,5,Rizki Juniansyah siap menjalani debut di Asian...,sport


### Cek Dataset

In [3]:
print("Jumlah data:", len(df))
print("\nJumlah missing value:")
print(df.isnull().sum())

print("\nDistribusi label:")
print(df["label"].value_counts())

print("\nJumlah data duplikat:", df.duplicated().sum())

Jumlah data: 200

Jumlah missing value:
id            0
isi_berita    0
label         0
dtype: int64

Distribusi label:
label
sport      100
finance    100
Name: count, dtype: int64

Jumlah data duplikat: 0


### Menentukan Kolom Teks

In [4]:
TEXT_COLUMN = "isi_berita"
TARGET_COLUMN = "label"

X = df[TEXT_COLUMN].fillna("").astype(str)
y = df[TARGET_COLUMN]

print("Kolom teks   :", TEXT_COLUMN)
print("Kolom target :", TARGET_COLUMN)
print("Jumlah teks  :", len(X))

print("\nContoh teks:")
print(X.iloc[0][:500])

Kolom teks   : isi_berita
Kolom target : label
Jumlah teks  : 200

Contoh teks:
Kejuaraan Nasional Gokart 2026 yang berlangsung dalam enam putaran menghadirkan persaingan ketat hingga seri terakhir. Romy Tahrizi berhasil menjadi Juara Nasional Shifter 150 tahun 2026 bersama Dewa United Motorsport setelah melalui perjuangan sepanjang musim. Kepastian tersebut diraih Romy pada Final Race Round 6 yang berlangsung pada 5-6 September 2026. Sebelumnya, pembalap yang juga berprofesi sebagai Lawyer di Kantor AHHN Lawyers tersebut masih menempati posisi kedua klasemen sementara deng


### Preprocessing Teks

In [5]:
def clean_text(text):
    text = text.lower()

    # Menghapus URL
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    # Menghapus HTML
    text = re.sub(r"<.*?>", " ", text)

    # Hanya mempertahankan huruf
    text = re.sub(r"[^a-z\s]", " ", text)

    # Menghapus spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["clean_text"] = df[TEXT_COLUMN].apply(clean_text)

print("Preprocessing berhasil.")
print("\nSebelum:")
print(df[TEXT_COLUMN].iloc[0][:300])

print("\nSesudah:")
print(df["clean_text"].iloc[0][:300])

Preprocessing berhasil.

Sebelum:
Kejuaraan Nasional Gokart 2026 yang berlangsung dalam enam putaran menghadirkan persaingan ketat hingga seri terakhir. Romy Tahrizi berhasil menjadi Juara Nasional Shifter 150 tahun 2026 bersama Dewa United Motorsport setelah melalui perjuangan sepanjang musim. Kepastian tersebut diraih Romy pada Fi

Sesudah:
kejuaraan nasional gokart yang berlangsung dalam enam putaran menghadirkan persaingan ketat hingga seri terakhir romy tahrizi berhasil menjadi juara nasional shifter tahun bersama dewa united motorsport setelah melalui perjuangan sepanjang musim kepastian tersebut diraih romy pada final race round y


### Tokenisasi

In [6]:
df["tokens"] = df["clean_text"].apply(lambda x: x.split())

print("Tokenisasi berhasil.")

print("\nContoh token:")
print(df["tokens"].iloc[0][:30])

print("\nJumlah token dokumen pertama:",
      len(df["tokens"].iloc[0]))

Tokenisasi berhasil.

Contoh token:
['kejuaraan', 'nasional', 'gokart', 'yang', 'berlangsung', 'dalam', 'enam', 'putaran', 'menghadirkan', 'persaingan', 'ketat', 'hingga', 'seri', 'terakhir', 'romy', 'tahrizi', 'berhasil', 'menjadi', 'juara', 'nasional', 'shifter', 'tahun', 'bersama', 'dewa', 'united', 'motorsport', 'setelah', 'melalui', 'perjuangan', 'sepanjang']

Jumlah token dokumen pertama: 618


### Train Test Split

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df[TARGET_COLUMN],
    test_size=0.20,
    random_state=42,
    stratify=df[TARGET_COLUMN]
)

print("Data training :", len(X_train))
print("Data testing  :", len(X_test))

print("\nDistribusi training:")
print(y_train.value_counts())

print("\nDistribusi testing:")
print(y_test.value_counts())

Data training : 160
Data testing  : 40

Distribusi training:
label
finance    80
sport      80
Name: count, dtype: int64

Distribusi testing:
label
finance    20
sport      20
Name: count, dtype: int64


### TF-IDF

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=None,
    ngram_range=(1, 1)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF berhasil dibuat.")

print("Ukuran matriks training :", X_train_tfidf.shape)
print("Ukuran matriks testing  :", X_test_tfidf.shape)
print("Jumlah vocabulary       :", len(tfidf.vocabulary_))

TF-IDF berhasil dibuat.
Ukuran matriks training : (160, 6240)
Ukuran matriks testing  : (40, 6240)
Jumlah vocabulary       : 6240


### Melihat Matriks TF-IDF

In [9]:
tfidf_feature_names = tfidf.get_feature_names_out()

tfidf_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=tfidf_feature_names
)

print("Ukuran tabel/matriks TF-IDF:")
print(tfidf_df.shape)

display(tfidf_df.iloc[:5, :20])

Ukuran tabel/matriks TF-IDF:
(160, 6240)


,aaron,abadi,abdul,abimanyu,abraham,absen,absorber,abu,ac,acara,acd,aceh,ach,achmad,acosta,across,activities,activity,ada,adalah
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.076124,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.02408,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.012265,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0


### Naive Bayes TF-IDF

In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

nb_tfidf = MultinomialNB()

nb_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)

accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)

print("Model Naive Bayes + TF-IDF berhasil dibuat.")
print("Accuracy TF-IDF:", accuracy_tfidf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tfidf))

Model Naive Bayes + TF-IDF berhasil dibuat.
Accuracy TF-IDF: 1.0

Classification Report:
              precision    recall  f1-score   support

     finance       1.00      1.00      1.00        20
       sport       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



### Membuat Skip-Gram

In [12]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 28.8 MB/s eta 0:00:00


In [30]:
from gensim.models import Word2Vec

sentences_train = [
    text.split() for text in X_train
]

skipgram = Word2Vec(
    sentences=sentences_train,
    vector_size=100,
    window=1,
    min_count=1,
    sg=1,
    workers=1,
    seed=42,
    epochs=10
)

print("Model Skip-Gram berhasil dibuat.")

print("Jumlah vocabulary:",
      len(skipgram.wv))

print("Ukuran vector:",
      skipgram.wv.vector_size)

Model Skip-Gram berhasil dibuat.
Jumlah vocabulary: 6263
Ukuran vector: 100


### Melihat Vektor Skip-Gram

In [31]:
words = list(skipgram.wv.index_to_key)

print("5 kata pertama:")
print(words[:5])

for word in words[:5]:
    print("\nKata:", word)
    print("Vector:", skipgram.wv[word])

5 kata pertama:
['dan', 'di', 'yang', 'untuk', 'ini']

Kata: dan
Vector: [-0.00847869  0.06901874 -0.10942101 -0.00438571 -0.15815794  0.23488224
  0.11729699  0.11691973 -0.11410295 -0.03798796 -0.16727656  0.33145005
  0.11734541  0.24462755 -0.04771006 -0.23455033  0.10795067 -0.49728733
 -0.08024513  0.02075068  0.01299406  0.16651694 -0.1883593   0.3311683
  0.22303112 -0.10200935 -0.26323953  0.18768176 -0.11046434 -0.0948609
  0.02432787 -0.09261078  0.17874865  0.01093681  0.01728912 -0.11805745
  0.24096173 -0.4215375  -0.09592813  0.16073684 -0.08927777 -0.0415126
  0.31255087 -0.04798459 -0.05790659  0.03701359  0.01255188  0.3238082
  0.42931134 -0.25420195 -0.14968443 -0.6609422   0.01869013 -0.04227096
  0.10256841 -0.09055133 -0.1266227   0.00942516 -0.06823453  0.2141425
 -0.00080171 -0.08314172 -0.3325268   0.16864161  0.17434213  0.05442747
  0.03254731 -0.12549882 -0.30325013 -0.02281097 -0.09741128 -0.16748513
 -0.1531833  -0.17021914  0.09660035 -0.03371751  0.0343

### Ukuran Matriks Skip-Gram

In [32]:
skipgram_matrix = skipgram.wv.vectors

print("Ukuran matriks Skip-Gram:")
print(skipgram_matrix.shape)

print("\nJumlah kata dalam vocabulary:",
      skipgram_matrix.shape[0])

print("Dimensi vector setiap kata:",
      skipgram_matrix.shape[1])

Ukuran matriks Skip-Gram:
(6263, 100)

Jumlah kata dalam vocabulary: 6263
Dimensi vector setiap kata: 100


### Membuat Vektor Dokumen dari Skip-Gram

In [33]:
def document_vector(text, model):
    words = text.split()

    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


X_train_skipgram = np.array([
    document_vector(text, skipgram)
    for text in X_train
])

X_test_skipgram = np.array([
    document_vector(text, skipgram)
    for text in X_test
])

print("Vektor dokumen Skip-Gram berhasil dibuat.")

print("Ukuran training:",
      X_train_skipgram.shape)

print("Ukuran testing:",
      X_test_skipgram.shape)

Vektor dokumen Skip-Gram berhasil dibuat.
Ukuran training: (160, 100)
Ukuran testing: (40, 100)


### Tabel Vektor Skip-Gram

In [34]:
embedding_cols = [f"Embedding_{i+1}" for i in range(X_train_skipgram.shape[1])]

skipgram_df = pd.DataFrame(
    X_train_skipgram,
    columns=embedding_cols
)

print("Ukuran tabel Skip-Gram:")
print(skipgram_df.shape)

display(skipgram_df.head(10))

Ukuran tabel Skip-Gram:
(160, 100)


,Embedding_1,Embedding_2,Embedding_3,Embedding_4,Embedding_5,Embedding_6,Embedding_7,Embedding_8,Embedding_9,Embedding_10,...,Embedding_91,Embedding_92,Embedding_93,Embedding_94,Embedding_95,Embedding_96,Embedding_97,Embedding_98,Embedding_99,Embedding_100
0,-0.003843,0.064657,-0.109907,0.039634,-0.155368,0.466044,0.229905,0.208923,-0.189207,0.124246,...,-0.182962,0.206672,0.237861,-0.131808,0.079415,-0.165210,0.017473,0.197008,-0.212568,-0.087693
1,0.021867,-0.022066,-0.057720,0.006144,-0.103318,0.176550,0.118409,0.127755,-0.078232,-0.006144,...,-0.133750,0.188047,0.084022,-0.140950,0.041874,-0.070450,0.029283,0.112280,-0.133079,-0.064705
2,-0.002891,0.064857,-0.111132,0.039769,-0.156395,0.470002,0.233127,0.212114,-0.190485,0.125641,...,-0.184655,0.209790,0.240678,-0.134332,0.080478,-0.167453,0.017052,0.199132,-0.214600,-0.089300
3,0.030714,-0.027999,-0.066334,0.007555,-0.120877,0.201573,0.135296,0.154759,-0.090789,-0.006821,...,-0.157750,0.218176,0.088319,-0.163079,0.052317,-0.079894,0.030728,0.127119,-0.152882,-0.073314
4,0.034700,-0.012568,-0.067298,0.011516,-0.113191,0.232950,0.139861,0.157023,-0.093325,0.034678,...,-0.157522,0.218805,0.102533,-0.127536,0.074315,-0.083465,0.026112,0.121623,-0.162028,-0.066216
5,0.007651,-0.020311,-0.062106,0.011065,-0.117511,0.211997,0.130218,0.152014,-0.090185,-0.005000,...,-0.158827,0.207929,0.099077,-0.150344,0.045048,-0.086482,0.038900,0.125940,-0.152763,-0.067073
6,0.025110,-0.021142,-0.055650,0.010441,-0.103134,0.194238,0.127325,0.142128,-0.082932,0.003584,...,-0.138794,0.190074,0.087087,-0.141561,0.048331,-0.080367,0.025939,0.111854,-0.132474,-0.066135
7,0.022284,-0.042007,-0.065312,0.006362,-0.120211,0.183437,0.132580,0.143730,-0.083924,-0.030519,...,-0.147648,0.210757,0.089408,-0.176987,0.044672,-0.067463,0.041035,0.130942,-0.144449,-0.076611
8,0.008482,-0.024741,-0.059582,0.011967,-0.108645,0.176672,0.117918,0.117668,-0.076840,-0.009527,...,-0.132549,0.174051,0.087229,-0.138778,0.044211,-0.059999,0.035827,0.112980,-0.130203,-0.068456
9,0.012515,-0.022776,-0.067672,0.020121,-0.106009,0.221159,0.134245,0.130177,-0.085794,0.015090,...,-0.136192,0.184531,0.104456,-0.136101,0.043920,-0.081063,0.023266,0.108341,-0.139613,-0.059166


### Perbandingan Ukuran TF-IDF dan Skip-Gram

In [35]:
comparison_space = pd.DataFrame({
    "Metode": [
        "TF-IDF",
        "Skip-Gram"
    ],
    "Jumlah Dokumen Training": [
        X_train_tfidf.shape[0],
        X_train_skipgram.shape[0]
    ],
    "Jumlah Fitur/Dimensi": [
        X_train_tfidf.shape[1],
        X_train_skipgram.shape[1]
    ],
    "Ukuran Matriks": [
        str(X_train_tfidf.shape),
        str(X_train_skipgram.shape)
    ]
})

display(comparison_space)

,Metode,Jumlah Dokumen Training,Jumlah Fitur/Dimensi,Ukuran Matriks
0,TF-IDF,160,6240,"(160, 6240)"
1,Skip-Gram,160,100,"(160, 100)"


### Naive Bayes Skip-Gram

In [36]:
nb_skipgram = MultinomialNB()

# Karena MultinomialNB membutuhkan nilai non-negatif,
# embedding dinormalisasi terlebih dahulu.
from sklearn.preprocessing import MinMaxScaler

scaler_skipgram = MinMaxScaler()

X_train_skipgram_scaled = scaler_skipgram.fit_transform(
    X_train_skipgram
)

X_test_skipgram_scaled = scaler_skipgram.transform(
    X_test_skipgram
)

nb_skipgram.fit(
    X_train_skipgram_scaled,
    y_train
)

y_pred_skipgram = nb_skipgram.predict(
    X_test_skipgram_scaled
)

accuracy_skipgram = accuracy_score(
    y_test,
    y_pred_skipgram
)

print("Model Naive Bayes + Skip-Gram berhasil.")
print("Accuracy Skip-Gram:", accuracy_skipgram)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_skipgram))

Model Naive Bayes + Skip-Gram berhasil.
Accuracy Skip-Gram: 0.825

Classification Report:
              precision    recall  f1-score   support

     finance       1.00      0.65      0.79        20
       sport       0.74      1.00      0.85        20

    accuracy                           0.82        40
   macro avg       0.87      0.82      0.82        40
weighted avg       0.87      0.82      0.82        40



### Perbandingan Akurasi

In [37]:
comparison_accuracy = pd.DataFrame({
    "Model": [
        "TF-IDF + Naive Bayes",
        "Skip-Gram + Naive Bayes"
    ],
    "Accuracy": [
        accuracy_tfidf,
        accuracy_skipgram
    ]
})

comparison_accuracy["Accuracy (%)"] = (
    comparison_accuracy["Accuracy"] * 100
)

display(comparison_accuracy)

,Model,Accuracy,Accuracy (%)
0,TF-IDF + Naive Bayes,1.000,100.0
1,Skip-Gram + Naive Bayes,0.825,82.5


### Grid Search TF-IDF + Naive Bayes

In [38]:
from sklearn.model_selection import GridSearchCV

param_grid_tfidf = {
    "alpha": [
        0.01,
        0.1,
        0.5,
        1.0,
        2.0
    ]
}

grid_tfidf = GridSearchCV(
    estimator=MultinomialNB(),
    param_grid=param_grid_tfidf,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_tfidf.fit(
    X_train_tfidf,
    y_train
)

print("Grid Search TF-IDF selesai.")
print("Best Parameter:",
      grid_tfidf.best_params_)

print("Best CV Score:",
      grid_tfidf.best_score_)

Grid Search TF-IDF selesai.
Best Parameter: {'alpha': 0.01}
Best CV Score: 0.9875


### Hasil Semua Grid Search TF-IDF

In [39]:
results_tfidf = pd.DataFrame(
    grid_tfidf.cv_results_
)

results_tfidf[
    [
        "param_alpha",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")

,param_alpha,mean_test_score,std_test_score,rank_test_score
0,0.01,0.98750,0.015309,1
1,0.10,0.98750,0.015309,1
2,0.50,0.98125,0.015309,3
3,1.00,0.98125,0.015309,3
4,2.00,0.97500,0.012500,5


### Grid Search Skip-Gram + Naive Bayes

In [40]:
param_grid_skipgram = {
    "alpha": [
        0.01,
        0.1,
        0.5,
        1.0,
        2.0
    ]
}

grid_skipgram = GridSearchCV(
    estimator=MultinomialNB(),
    param_grid=param_grid_skipgram,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_skipgram.fit(
    X_train_skipgram_scaled,
    y_train
)

print("Grid Search Skip-Gram selesai.")

print("Best Parameter:",
      grid_skipgram.best_params_)

print("Best CV Score:",
      grid_skipgram.best_score_)

Grid Search Skip-Gram selesai.
Best Parameter: {'alpha': 0.01}
Best CV Score: 0.76875


### Hasil Grid Search Skip-Gram

In [41]:
results_skipgram = pd.DataFrame(
    grid_skipgram.cv_results_
)

display(
    results_skipgram[
        [
            "param_alpha",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ].sort_values("rank_test_score")
)

,param_alpha,mean_test_score,std_test_score,rank_test_score
0,0.01,0.76875,0.046771,1
1,0.10,0.76875,0.046771,1
2,0.50,0.76250,0.042390,3
3,1.00,0.73750,0.037500,4
4,2.00,0.72500,0.036443,5


### Evaluasi Model Terbaik

In [42]:
best_tfidf_model = grid_tfidf.best_estimator_
best_skipgram_model = grid_skipgram.best_estimator_

pred_best_tfidf = best_tfidf_model.predict(
    X_test_tfidf
)

pred_best_skipgram = best_skipgram_model.predict(
    X_test_skipgram_scaled
)

acc_best_tfidf = accuracy_score(
    y_test,
    pred_best_tfidf
)

acc_best_skipgram = accuracy_score(
    y_test,
    pred_best_skipgram
)

print("Accuracy terbaik TF-IDF:",
      acc_best_tfidf)

print("Accuracy terbaik Skip-Gram:",
      acc_best_skipgram)

Accuracy terbaik TF-IDF: 1.0
Accuracy terbaik Skip-Gram: 0.825


### Tabel Perbandingan Akhir

In [43]:
final_comparison = pd.DataFrame({
    "Model": [
        "TF-IDF + Naive Bayes",
        "Skip-Gram + Naive Bayes"
    ],
    "Best Alpha": [
        grid_tfidf.best_params_["alpha"],
        grid_skipgram.best_params_["alpha"]
    ],
    "CV Accuracy": [
        grid_tfidf.best_score_,
        grid_skipgram.best_score_
    ],
    "Test Accuracy": [
        acc_best_tfidf,
        acc_best_skipgram
    ]
})

final_comparison["Test Accuracy (%)"] = (
    final_comparison["Test Accuracy"] * 100
)

display(final_comparison)

,Model,Best Alpha,CV Accuracy,Test Accuracy,Test Accuracy (%)
0,TF-IDF + Naive Bayes,0.01,0.98750,1.000,100.0
1,Skip-Gram + Naive Bayes,0.01,0.76875,0.825,82.5


### Classification Report Model Terbaik

In [44]:
print("=== TF-IDF + NAIVE BAYES ===")
print(
    classification_report(
        y_test,
        pred_best_tfidf
    )
)

print("\n=== SKIP-GRAM + NAIVE BAYES ===")
print(
    classification_report(
        y_test,
        pred_best_skipgram
    )
)

=== TF-IDF + NAIVE BAYES ===
              precision    recall  f1-score   support

     finance       1.00      1.00      1.00        20
       sport       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40


=== SKIP-GRAM + NAIVE BAYES ===
              precision    recall  f1-score   support

     finance       1.00      0.65      0.79        20
       sport       0.74      1.00      0.85        20

    accuracy                           0.82        40
   macro avg       0.87      0.82      0.82        40
weighted avg       0.87      0.82      0.82        40



### Confusion Matrix

In [45]:
from sklearn.metrics import confusion_matrix

cm_tfidf = confusion_matrix(
    y_test,
    pred_best_tfidf
)

cm_skipgram = confusion_matrix(
    y_test,
    pred_best_skipgram
)

print("Confusion Matrix TF-IDF:")
print(cm_tfidf)

print("\nConfusion Matrix Skip-Gram:")
print(cm_skipgram)

Confusion Matrix TF-IDF:
[[20  0]
 [ 0 20]]

Confusion Matrix Skip-Gram:
[[13  7]
 [ 0 20]]


### Menentukan Model Skip-Gram Final

In [46]:
final_skipgram_model = {
    "word2vec": skipgram,
    "scaler": scaler_skipgram,
    "classifier": best_skipgram_model,
    "vector_size": 3,
    "window": 1,
    "sg": 1,
    "seed": 42
}

print("Komponen model Skip-Gram:")
print(final_skipgram_model.keys())

Komponen model Skip-Gram:
dict_keys(['word2vec', 'scaler', 'classifier', 'vector_size', 'window', 'sg', 'seed'])


### Save Model .pkl

In [47]:
import pickle

with open("skipgram_model.pkl", "wb") as f:
    pickle.dump(skipgram, f)

print("Model Skip-Gram berhasil disimpan.")
print("File: skipgram_model.pkl")

with open("naive_bayes_model.pkl", "wb") as f:
    pickle.dump(best_skipgram_model, f)

print("Model Naive Bayes berhasil disimpan.")
print("File: naive_bayes_model.pkl")

with open("scaler_skipgram.pkl", "wb") as f:
    pickle.dump(scaler_skipgram, f)

print("Scaler Skip-Gram berhasil disimpan.")
print("File: scaler_skipgram.pkl")

with open("skipgram_model.pkl", "rb") as f:
    loaded_skipgram = pickle.load(f)

with open("naive_bayes_model.pkl", "rb") as f:
    loaded_nb = pickle.load(f)

with open("scaler_skipgram.pkl", "rb") as f:
    loaded_scaler = pickle.load(f)

print("Semua model berhasil di-load.")

print("\nSkip-Gram:")
print(type(loaded_skipgram))

print("\nNaive Bayes:")
print(type(loaded_nb))

print("\nScaler:")
print(type(loaded_scaler))

Model Skip-Gram berhasil disimpan.
File: skipgram_model.pkl
Model Naive Bayes berhasil disimpan.
File: naive_bayes_model.pkl
Scaler Skip-Gram berhasil disimpan.
File: scaler_skipgram.pkl
Semua model berhasil di-load.

Skip-Gram:
<class 'gensim.models.word2vec.Word2Vec'>

Naive Bayes:
<class 'sklearn.naive_bayes.MultinomialNB'>

Scaler:
<class 'sklearn.preprocessing._data.MinMaxScaler'>
